Direct Preference Optimization (DPO) for LLM Alignment (From Scratch)

In [ ]:
# 环境自检:导入 importlib.metadata.version,用于查询已安装第三方库的版本号
from importlib.metadata import version

# 定义需要检查版本的关键库列表: tiktoken(GPT-2 BPE 分词器)与 torch(深度学习框架)
pkgs = [
    "tiktoken",    # Tokenizer
    "torch",       # Deep learning library
]
# 遍历列表并打印每个库的版本,确保后续 DPO 训练代码所需依赖已正确安装
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# 下载(或读取本地缓存的)DPO 偏好数据集: 每条样本包含 instruction/input,
# 以及一对 chosen(更优)/rejected(较差)回答,是 DPO 训练的核心数据来源
import json
import os
import requests


# 辅助函数:若本地文件不存在则联网下载并缓存到磁盘,存在则直接读取,避免重复下载
def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    # 将下载/读取到的 JSON 文本字符串解析为 Python 对象(list[dict]),每个 dict 是一条偏好样本
    data = json.loads(text_data)
    return data


# 数据集本地缓存路径与远程下载地址(GitHub 原始文件)
file_path = "instruction-data-with-preference.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/04_preference-tuning-with-dpo/instruction-data-with-preference.json"
)

# 调用上面的函数,得到完整的偏好数据集列表
data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

In [ ]:
# 用 pprint 美化打印,查看第 50 条样本的原始结构(instruction/input/chosen/rejected 等字段)
import pprint

pprint.pp(data[50])

In [ ]:
# 再看第 999 条样本,直观感受 chosen 与 rejected 回答之间的质量差异
pprint.pp(data[999])

In [ ]:
# 将一条样本的 instruction(+可选 input)拼接成 Alpaca 风格的提示词模板,与 SFT 阶段保持一致
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    # 若样本包含 input 字段(附加上下文),则拼接 ### Input 部分;否则为空字符串
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [ ]:
# 用第 50 条样本验证 format_input 的输出效果
model_input = format_input(data[50])
print(model_input)

In [ ]:
# 拼接 chosen(偏好/更优)回答,作为 DPO 中希望模型概率更高的目标回答
desired_response = f"### Response:\n{data[50]['chosen']}"
print(desired_response)

In [ ]:
# 拼接 rejected(较差)回答,作为 DPO 中希望模型概率降低的回答
possible_response = f"### Response:\n{data[50]['rejected']}"
print(possible_response)

In [ ]:
# 按 85% / 10% / 5% 的比例切分训练集、测试集与验证集(按顺序切分,非随机打乱)
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]
print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

In [ ]:
# 定义 PyTorch Dataset:对每条偏好样本预先做好 prompt/chosen/rejected 三种文本的分词(tokenize),
# 训练时直接取用,避免每个 epoch 重复编码
import torch
from torch.utils.data import Dataset


class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        # 依次处理数据集中的每一条样本
        for entry in data:
            prompt = format_input(entry)
            rejected_response = entry["rejected"]
            chosen_response = entry["chosen"]

            # 仅对 prompt 部分单独分词,用于后续构造 mask(标记哪些位置属于提示词,不计入损失)
            prompt_tokens = tokenizer.encode(prompt)
            # 分别拼接“提示词 + 完整回答”文本: chosen_full_text 对应偏好回答,rejected_full_text 对应较差回答
            chosen_full_text = f"{prompt}\n\n### Response:\n{chosen_response}"
            rejected_full_text = f"{prompt}\n\n### Response:\n{rejected_response}"
            # 对完整文本(prompt+response)分别分词,得到用于送入模型的 token id 序列
            chosen_full_tokens = tokenizer.encode(chosen_full_text)
            rejected_full_tokens = tokenizer.encode(rejected_full_text)

            # 保存该样本的三组 token id 列表,供 __getitem__ 按索引取出
            self.encoded_texts.append({
                "prompt": prompt_tokens,
                "chosen": chosen_full_tokens,
                "rejected": rejected_full_tokens,
            })

    # Dataset 标准接口:按索引返回预先分词好的样本(dict 形式)
    def __getitem__(self, index):
        return self.encoded_texts[index]

    # Dataset 标准接口:返回数据集样本总数
    def __len__(self):
        return len(self.data)

In [ ]:
# 自定义 collate_fn:把一个 batch 内长度不同的样本 padding 对齐,并生成 chosen/rejected 的掩码(mask)
# 输出张量形状均为 (batch_size, max_length_common),max_length_common 由本 batch 内最长序列决定
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    allowed_max_length=None,
    mask_prompt_tokens=True,
    device="cpu"
):
    # Initialize lists to hold batch data
    batch_data = {
        "prompt": [],
        "chosen": [],
        "rejected": [],
        "rejected_mask": [],
        "chosen_mask": []

    }

    # 分别统计 chosen 和 rejected 两组序列的最大长度(+1 为后面 label 右移预留一个位置),取二者的最大值作为公共 padding 长度
    # Determine the longest sequence to set a common padding length
    max_length_common = 0
    if batch:
        for key in ["chosen", "rejected"]:
            current_max = max(len(item[key])+1 for item in batch)
            max_length_common = max(max_length_common, current_max)

    # 遍历 batch 中的每个样本,对 chosen/rejected 做 padding 并构造对应的 mask
    # Process each item in the batch
    for item in batch:
        prompt = torch.tensor(item["prompt"])
        batch_data["prompt"].append(prompt)

        for key in ["chosen", "rejected"]:
            # 用 pad_token_id 把序列右侧补齐到 max_length_common,mask 初始全部为 True(表示都参与计算)
            # Adjust padding according to the common maximum length
            sequence = item[key]
            padded = sequence + [pad_token_id] * (max_length_common - len(sequence))
            mask = torch.ones(len(padded)).bool()

            # 把 padding 部分对应位置的 mask 置为 False,避免 padding token 参与 log 概率的计算
            # Set mask for all padding tokens to False
            mask[len(sequence):] = False

            # 若开启 mask_prompt_tokens,则连提示词(prompt)部分也置为 False,只让“回答”部分参与 DPO 损失
            # Set mask for all input tokens to False
            # +2 sets the 2 newline ("\n") tokens before "### Response" to False
            if mask_prompt_tokens:
                mask[:prompt.shape[0]+2] = False

            batch_data[key].append(torch.tensor(padded))
            batch_data[f"{key}_mask"].append(mask)

    # 把每个 key 对应的样本列表堆叠(stack)成一个 batch 张量: 形状 (batch_size, max_length_common)
    # Final processing
    for key in ["chosen", "rejected", "chosen_mask", "rejected_mask"]:
        # Stack all sequences into a tensor for the given key
        tensor_stack = torch.stack(batch_data[key])

        # 若指定了 allowed_max_length,则在序列维度上做截断,防止超出模型支持的最大上下文长度
        # Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            tensor_stack = tensor_stack[:, :allowed_max_length]

        # 将张量搬到目标设备(如 GPU),便于后续训练直接使用
        # Move to the specified device
        batch_data[key] = tensor_stack.to(device)

    return batch_data

In [ ]:
# 自动选择可用的计算设备:优先 CUDA GPU,其次 Apple Silicon 的 MPS(需较新版本 PyTorch 才稳定),否则退回 CPU
from functools import partial

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("Device:", device)

# 用 functools.partial 预先固定 custom_collate_fn 的部分参数,得到 DataLoader 可直接使用的 collate_fn
customized_collate_fn = partial(
    custom_collate_fn,
    device=device,            # Put the data directly on a GPU if available
    mask_prompt_tokens=True,  # This is optional
    allowed_max_length=1024   # The supported context length of the model
)

In [ ]:
# 取前两条样本做小规模演示,便于观察数据处理流程与张量形状
example_data = data[:2]

for i in example_data:
    print()
    pprint.pp(i)

In [ ]:
# 加载 GPT-2 的 BPE 分词器(tiktoken),用于把文本转换为模型可处理的 token id 序列
import tiktoken
from torch.utils.data import DataLoader


tokenizer = tiktoken.get_encoding("gpt2")

example_dataset = PreferenceDataset(example_data, tokenizer)

# 用示例数据 + 自定义 collate_fn 构建一个小型 DataLoader,方便下面演示 batch 的结构与形状
example_dataloader = DataLoader(
    example_dataset,
    batch_size=2,
    collate_fn=customized_collate_fn,
    shuffle=False
)

In [ ]:
# 取出 DataLoader 产出的第一个 batch(仅用于查看结构,不遍历全部)
for batch in example_dataloader:
    break

# batch 是一个 dict,包含 prompt/chosen/rejected/chosen_mask/rejected_mask 五个键,每个值都是张量
print("batch.keys:", batch.keys())

In [ ]:
# 辅助函数:把 batch 中某个样本的 token id 张量展平(flatten)后解码回可读文本,便于人工检查数据是否正确
def decode_tokens_from_batch(token_ids, tokenizer):
    ids_in_python_list = token_ids.flatten().tolist()
    return tokenizer.decode(ids_in_python_list)

In [ ]:
# 查看 batch 中第一条样本的 prompt 部分被还原成什么文本
text = decode_tokens_from_batch(
    token_ids=batch["prompt"][0],  # [0] for the first entry in the batch
    tokenizer=tokenizer,
)
print(text)

In [ ]:
# 查看第一条样本的 chosen(完整 prompt + 偏好回答)文本
text = decode_tokens_from_batch(
    token_ids=batch["chosen"][0],
    tokenizer=tokenizer,
)
print(text)

In [ ]:
# 查看第一条样本的 rejected(完整 prompt + 较差回答)文本
text = decode_tokens_from_batch(
    token_ids=batch["rejected"][0],
    tokenizer=tokenizer,
)
print(text)

In [ ]:
# 打印 chosen 序列与其 mask 的形状,二者长度一致,形状均为 (seq_len,)
print("chosen inputs:", batch["chosen"][0].shape)
print("chosen mask:  ", batch["chosen_mask"][0].shape)
batch["chosen_mask"][0]

In [ ]:
# 用 mask 过滤掉 prompt 和 padding 部分,只保留参与损失计算的 chosen 回答 token,验证 mask 是否正确
text = decode_tokens_from_batch(
    token_ids=batch["chosen"][0][batch["chosen_mask"][0]],
    tokenizer=tokenizer,
)
print(text)

In [ ]:
# 同样验证 rejected 回答的 mask 是否正确地只保留了回答部分
text = decode_tokens_from_batch(
    token_ids=batch["rejected"][0][batch["rejected_mask"][0]],
    tokenizer=tokenizer,
)
print(text)

In [ ]:
# 正式构建训练/验证/测试三个 Dataset 与 DataLoader
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

# 固定随机种子,保证 shuffle 等随机行为可复现
torch.manual_seed(123)

# 训练集: 开启 shuffle 打乱样本顺序,drop_last=True 丢弃不满一个 batch 的尾部数据,保证每个 batch 大小一致
train_dataset = PreferenceDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)
# 验证集: 不打乱、不丢弃尾部数据,便于完整评估
val_dataset = PreferenceDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

# 测试集: 同验证集,保留全部样本用于最终评估
test_dataset = PreferenceDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [ ]:
# 遍历训练 DataLoader,查看每个 batch 中 chosen/rejected 张量的形状: (batch_size, 该 batch 的公共 padding 长度)
print("Train loader:")
for batch in train_loader:
    print(
        batch["chosen"].shape,
        batch["rejected"].shape,
    )

In [ ]:
# 尝试定位第 7 章 SFT(监督微调)阶段保存的模型权重文件 gpt2-medium355M-sft.pth
# DPO 需要在一个已完成指令微调(SFT)的模型基础上继续训练,因此必须先拿到该权重
from pathlib import Path
import shutil


finetuned_model_path = Path("gpt2-medium355M-sft.pth")
if not finetuned_model_path.exists():

    # 优先尝试从上一章节(01_main-chapter-code)的输出目录中拷贝权重文件
    # Try finding the model checkpoint locally:
    relative_path = Path("..") / "01_main-chapter-code" / finetuned_model_path
    if relative_path.exists():
        shutil.copy(relative_path, ".")

    # 若在 Google Colab 环境运行,则改为从挂载的 Google Drive 中拷贝权重(需读者自行调整路径)
    # If this notebook is run on Google Colab, get it from a Google Drive folder
    elif "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
        google_drive_path = "/content/drive/My Drive/Books/LLMs-From-Scratch/ch07/colab/gpt2-medium355M-sft.pth"  # Readers need to adjust this path
        shutil.copy(google_drive_path, ".")

    # 都找不到时,提示用户先运行 ch07.ipynb 完成 SFT 微调并保存模型
    else:
        print(
            f"Could not find '{finetuned_model_path}'.\n"
            "Run the `ch07.ipynb` notebook to finetune and save the finetuned model."
        )

In [ ]:
# 从本地 previous_chapters.py 导入第 4 章实现的 GPTModel 架构类
from previous_chapters import GPTModel
# If the `previous_chapters.py` file is not available locally,
# you can import it from the `llms-from-scratch` PyPI package.
# For details, see: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# E.g.,
# from llms_from_scratch.ch04 import GPTModel


# 模型的基础超参数配置(词表大小、上下文长度、dropout、是否使用 QKV 偏置)
BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

# 不同规模 GPT-2 模型对应的结构超参数(embedding 维度、层数、注意力头数)
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# 本 notebook 选用 gpt2-medium (355M) 作为基座模型
CHOOSE_MODEL = "gpt2-medium (355M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

# 用最终配置实例化模型结构,并加载第 7 章 SFT 阶段训练好的权重(不是随机初始化)
model = GPTModel(BASE_CONFIG)
model.load_state_dict(
    torch.load(
        "gpt2-medium355M-sft.pth",
        map_location=torch.device("cpu"),
        weights_only=True
    )
)
# 切换为 eval 模式(关闭 dropout 等训练专用行为),用于后续生成测试
model.eval();

In [ ]:
# 构造一个测试用的指令 prompt,验证 SFT 后模型的生成效果,作为 DPO 训练前的基线
prompt = """Below is an instruction that describes a task. Write a response
that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'
"""
# 从第 5 章引入生成相关的辅助函数: generate(自回归采样生成)、
# text_to_token_ids/token_ids_to_text(文本与 token id 互转)
from previous_chapters import (
    generate,
    text_to_token_ids,
    token_ids_to_text
)
# 注意: 下面这行替代导入写法的注释有笔误,正确应为
# `from llms_from_scratch.ch05 import (`(缺少 import 关键字);
# 这只是一段说明性注释,不是会被执行的代码,不影响实际运行(bug 旁注)
# Alternatively:
# from llms_from_scratch.ch05 (
#     generate,
#     text_to_token_ids,
#     token_ids_to_text
# )

# 固定随机种子以保证生成结果可复现
torch.manual_seed(123)

# 用当前模型(此时即 SFT 权重)自回归生成最多 35 个新 token,直到遇到 eos_id 或达到长度上限
token_ids = generate(
    model=model,
    idx=text_to_token_ids(prompt, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256
)

response = token_ids_to_text(token_ids, tokenizer)
print(response)

In [ ]:
# 从模型生成的完整文本中截取出“### Response:”之后、去除模板标记的纯回答部分
def extract_response(response_text, input_text):
    return response_text[len(input_text):].replace("### Response:", "").strip()

response = extract_response(response, prompt)
print(response)

In [ ]:
# DPO 需要两个模型: policy_model(待优化的策略模型),直接复用刚才微调好的 model
policy_model = model

# reference_model(参考/冻结模型): 重新实例化并加载同样的 SFT 权重,训练过程中其参数始终不变,
# 用作 DPO 损失中约束 policy_model 不要偏离太远的“锚点”
reference_model = GPTModel(BASE_CONFIG)
reference_model.load_state_dict(
    torch.load(
        "gpt2-medium355M-sft.pth",
        map_location=torch.device("cpu"),
        weights_only=True
    )
)
# 参考模型全程只用于前向推理,不参与反向传播,故设为 eval 模式
reference_model.eval()

# 将两个模型都搬到计算设备(GPU/MPS/CPU)上
policy_model.to(device)
reference_model.to(device);

In [ ]:
# 核心:实现 DPO(Direct Preference Optimization)损失函数
import torch.nn.functional as F

# 四个输入均为形状 (batch_size,) 的对数概率(每个样本一个标量): 分别是 policy 模型和 reference 模型
# 在 chosen(偏好)回答和 rejected(较差)回答上的平均 log 概率
def compute_dpo_loss(
      model_chosen_logprobs,
      model_rejected_logprobs,
      reference_chosen_logprobs,
      reference_rejected_logprobs,
      beta=0.1,
    ):
    """Compute the DPO loss for a batch of policy and reference model log probabilities.

    Args:
        model_chosen_logprobs: Log probabilities of the policy model for the chosen responses. Shape: (batch_size,)
        model_rejected_logprobs: Log probabilities of the policy model for the rejected responses. Shape: (batch_size,)
        reference_chosen_logprobs: Log probabilities of the reference model for the chosen responses. Shape: (batch_size,)
        reference_rejected_logprobs: Log probabilities of the reference model for the rejected responses. Shape: (batch_size,)
        beta: Temperature parameter for the DPO loss; typically something in the range of 0.1 to 0.5. We ignore the reference model as beta -> 0.

    Returns:
        A tuple of three tensors: (loss, chosen_rewards, rejected_rewards).
    """

    # model_logratios: policy 模型对 (chosen 相对 rejected) 的偏好程度,即 log π(chosen) - log π(rejected)
    model_logratios = model_chosen_logprobs - model_rejected_logprobs
    # reference_logratios: reference(冻结)模型对同一对回答的偏好程度,作为对照基准
    reference_logratios = reference_chosen_logprobs - reference_rejected_logprobs
    # logits: policy 相对 reference 的“偏好提升量”,即 (policy 的偏好差) - (reference 的偏好差);
    # 该值越大,说明相对参考模型,policy 模型越倾向于选择 chosen 而非 rejected
    logits = model_logratios - reference_logratios

    # DPO 损失公式: loss = -log(sigmoid(beta * logits))
    # beta 是温度系数,控制允许 policy 偏离 reference 的程度: beta 越小,惩罚越弱,policy 可以更自由地偏离参考模型;
    # beta 越大,则越强迫 policy 保持接近 reference;beta -> 0 时该损失几乎忽略参考模型的作用
    # DPO (Eq. 7 of https://arxiv.org/pdf/2305.18290.pdf)
    losses = -F.logsigmoid(beta * logits)

    # chosen_rewards / rejected_rewards 是训练过程中用于监控的“隐式奖励”指标(policy 相对 reference 的 log 概率差),
    # 用 .detach() 断开梯度,不参与反向传播,仅用于打印/记录
    # Optional values to track progress during training
    chosen_rewards = (model_chosen_logprobs - reference_chosen_logprobs).detach()
    rejected_rewards = (model_rejected_logprobs - reference_rejected_logprobs).detach()

    # 对 batch 内所有样本的标量损失取平均,得到最终标量 loss;chosen/rejected_rewards 同样在 batch 维度上取平均
    # .mean() to average over the samples in the batch
    return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()

In [ ]:
# 计算给定 token 序列在语言模型下的(平均)log 概率,是 compute_dpo_loss 的输入来源
def compute_logprobs(logits, labels, selection_mask=None):
    """
    Compute log probabilities.

    Args:
      logits: Tensor of shape (batch_size, num_tokens, vocab_size)
      labels: Tensor of shape (batch_size, num_tokens)
      selection_mask: Tensor for shape (batch_size, num_tokens)

    Returns:
      mean_log_prob: Mean log probability excluding padding tokens.
    """

    # 语言模型是自回归的: 第 t 个位置的 logits 用来预测第 t+1 个 token,因此真实标签要整体左移一位(去掉第一个 token)
    # labels 形状从 (batch_size, num_tokens) 变为 (batch_size, num_tokens-1)
    # Labels are the inputs shifted by one
    labels = labels[:, 1:].clone()

    # 相应地丢弃最后一个位置的 logits(它预测的是序列之后不存在的 token),使 logits 与 labels 在序列长度上对齐
    # logits 形状从 (batch_size, num_tokens, vocab_size) 变为 (batch_size, num_tokens-1, vocab_size)
    # Truncate logits to match the labels num_tokens
    logits = logits[:, :-1, :]

    # 对词表维度做 log_softmax,得到每个位置上每个词表 token 的 log 概率,形状不变
    log_probs = F.log_softmax(logits, dim=-1)

    # 用 torch.gather 按 labels 中的真实 token id,从 log_probs 里取出对应位置“实际生成的那个 token”的 log 概率;
    # index 需先 unsqueeze(-1) 变成 (batch_size, num_tokens-1, 1) 才能在最后一维 gather,gather 后再 squeeze(-1) 还原;
    # 结果 selected_log_probs 形状为 (batch_size, num_tokens-1)
    # Gather the log probabilities for the actual labels
    selected_log_probs = torch.gather(
        input=log_probs,
        dim=-1,
        index=labels.unsqueeze(-1)
    ).squeeze(-1)

    # 若提供了 selection_mask(标记哪些位置是有效回答 token,哪些是 prompt/padding),同样左移一位对齐
    if selection_mask is not None:
        mask = selection_mask[:, 1:].clone()

        # 把 mask 为 False 的位置(prompt 部分和 padding 部分)的 log 概率置零,使其不参与后续求和
        # Apply the mask to filter out padding tokens
        selected_log_probs = selected_log_probs * mask

        # 对 token 维度求和后再除以有效 token 数(mask.sum(-1)),得到排除 padding/prompt 影响的“平均每 token log 概率”
        # Calculate the average log probability excluding padding tokens
        # This averages over the tokens, so the shape is (batch_size,)
        avg_log_prob = selected_log_probs.sum(-1) / mask.sum(-1)

        return avg_log_prob

    # 未提供 mask 时,直接对所有位置取平均(不推荐用于变长 padding 场景)
    else:
        return selected_log_probs.mean(-1)

In [ ]:
# 用一个简化的小例子验证:上面 compute_logprobs 里 log_softmax + gather 的手工实现,
# 其本质等价于 PyTorch 内置的交叉熵损失 F.cross_entropy(仅差符号与平均方式的对应关系)
# Sample data
logits = torch.tensor(
    [[2.0, 1.0, 0.1],
     [0.5, 2.5, 0.3]])  # Shape: (2, 3)
targets = torch.tensor([0, 2])  # Shape: (2,)


# 手动实现: 先对 logits 做 log_softmax,再用 gather 取出每个样本在其目标类别上的 log 概率,取负号并平均即为交叉熵损失
# Manual loss using torch.gather
log_softmax_logits = F.log_softmax(logits, dim=1)  # Shape: (2, 3)
selected_log_probs = torch.gather(
    input=log_softmax_logits,
    dim=1,
    index=targets.unsqueeze(1), # Shape 2, 1
).squeeze(1)  # Shape: (2,)
manual_loss = -selected_log_probs.mean()  # Averaging over the batch


# 直接调用 PyTorch 自带的 cross_entropy 做对比,验证二者数值应当一致
# PyTorch loss
cross_entropy_loss = F.cross_entropy(logits, targets)

print(manual_loss, cross_entropy_loss)

In [ ]:
# 构造一个更简单的例子,进一步说明 torch.gather 的用法:按 index 中给出的下标,从 t 的每一行中取值
t = torch.tensor(
  [[1., 2.,],
   [3., 4.]]
)

m = torch.tensor(
  [[1, 1],
   [0, 1]]
)

In [ ]:
# index=[[1,1],[0,1]] 表示: 第 0 行取 t[0] 中下标 1、1 位置的值 -> [2., 2.];第 1 行取 t[1] 中下标 0、1 位置的值 -> [3., 4.]
torch.gather(input=t, dim=-1, index=m)

In [ ]:
# 整合以上组件: 对一个 batch,分别计算 policy 模型和 reference 模型在 chosen/rejected 上的 log 概率,再喂给 compute_dpo_loss
def compute_dpo_loss_batch(batch, policy_model, reference_model, beta):
    """Compute the DPO loss on an input batch"""

    # where policy_model(batch["chosen"]) are the logits
    # policy_model 对 chosen 回答做前向传播,得到 logits(形状 (batch_size, seq_len, vocab_size)),再转成平均 log 概率 (batch_size,)
    policy_chosen_log_probas = compute_logprobs(
        logits=policy_model(batch["chosen"]),
        labels=batch["chosen"],
        selection_mask=batch["chosen_mask"]
    )
    # 同理计算 policy_model 在 rejected 回答上的平均 log 概率
    policy_rejected_log_probas = compute_logprobs(
        logits=policy_model(batch["rejected"]),
        labels=batch["rejected"],
        selection_mask=batch["rejected_mask"]
    )

    # reference 模型全程冻结,用 torch.no_grad() 包裹以节省显存、避免计算不必要的梯度
    with torch.no_grad():
        # 分别计算 reference 模型在 chosen / rejected 回答上的平均 log 概率,作为 DPO 损失中的基准项
        ref_chosen_log_probas = compute_logprobs(
            logits=reference_model(batch["chosen"]),
            labels=batch["chosen"],
            selection_mask=batch["chosen_mask"]
        )
        ref_rejected_log_probas = compute_logprobs(
            logits=reference_model(batch["rejected"]),
            labels=batch["rejected"],
            selection_mask=batch["rejected_mask"]
        )
    # 将四组 log 概率一起送入 compute_dpo_loss,得到最终标量损失以及用于监控的 chosen/rejected 隐式奖励
    loss, chosen_rewards, rejected_rewards = compute_dpo_loss(
        model_chosen_logprobs=policy_chosen_log_probas,
        model_rejected_logprobs=policy_rejected_log_probas,
        reference_chosen_logprobs=ref_chosen_log_probas,
        reference_rejected_logprobs=ref_rejected_log_probas,
        beta=beta
    )
    return loss, chosen_rewards, rejected_rewards

In [ ]:
# 用前面示例 batch 快速验证 compute_dpo_loss_batch 能正常跑通(此处仅前向计算,不做反向传播)
with torch.no_grad():
    loss = compute_dpo_loss_batch(batch, policy_model, reference_model, beta=0.1)
print(loss)

In [ ]:
# 对整个 DataLoader(可指定只跑前 num_batches 个 batch)累加 loss/chosen_rewards/rejected_rewards,用于评估阶段
def compute_dpo_loss_loader(data_loader, policy_model, reference_model, beta, num_batches=None):
    """Apply compute_dpo_loss_batch to a whole data loader"""

    total_loss, total_chosen_rewards, total_rejected_rewards = 0., 0., 0.
    if len(data_loader) == 0:
        return float("nan")

    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # 若指定的 num_batches 超过了实际 batch 总数,则取二者较小值,避免越界
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    # 逐 batch 计算 DPO 损失,累加损失和奖励指标,凑够 num_batches 个 batch 后提前退出
    for i, batch in enumerate(data_loader):
        if i < num_batches:
            loss, chosen_rewards, rejected_rewards = compute_dpo_loss_batch(
                batch=batch,
                policy_model=policy_model,
                reference_model=reference_model,
                beta=beta
            )
            total_loss += loss.item()
            total_chosen_rewards += chosen_rewards.item()
            total_rejected_rewards += rejected_rewards.item()

        else:
            break

    # 对累加值求平均,得到该 loader 上的平均损失与平均奖励
    # calculate average
    total_loss /= num_batches
    total_chosen_rewards /= num_batches
    total_rejected_rewards /= num_batches
    return total_loss, total_chosen_rewards, total_rejected_rewards

In [ ]:
# 训练过程中周期性调用: 分别在训练集和验证集上采样 eval_iter 个 batch,评估当前 policy 模型的 DPO 损失与奖励margin
def evaluate_dpo_loss_loader(policy_model, reference_model, train_loader, val_loader, beta, eval_iter):
    """Compute the DPO loss for the training and validation dataset"""

    # 评估前切到 eval 模式(关闭 dropout),并用 no_grad 关闭梯度以节省显存、加快计算
    policy_model.eval()
    with torch.no_grad():
        train_loss, train_chosen_rewards, train_rejected_rewards = compute_dpo_loss_loader(
            data_loader=train_loader,
            policy_model=policy_model,
            reference_model=reference_model,
            beta=beta,
            num_batches=eval_iter
        )

        val_loss, val_chosen_rewards, val_rejected_rewards = compute_dpo_loss_loader(
            data_loader=val_loader,
            policy_model=policy_model,
            reference_model=reference_model,
            beta=beta,
            num_batches=eval_iter
        )

    res = {
    # 汇总训练/验证两部分的损失与奖励指标,打包成字典返回
        "train_loss": train_loss,
        "train_chosen_reward": train_chosen_rewards,
        "train_rejected_reward": train_rejected_rewards,
        "val_loss": val_loss,
        "val_chosen_reward": val_chosen_rewards,
        "val_rejected_reward": val_rejected_rewards
    }

    policy_model.train()
    # 评估结束后把 policy_model 切回训练模式,不影响后续的训练循环
    return res

In [ ]:
# 从第 5 章引入 generate_and_print_sample: 每个 epoch 结束后用当前模型生成一段示例文本,直观查看训练效果
from previous_chapters import generate_and_print_sample
# Alternatively:
# from llms_from_scratch.ch04 import generate_text_simple


# DPO 训练主循环: 每个 step 计算 policy 相对 reference 的 DPO 损失并反向传播更新 policy_model,
# reference_model 全程冻结不更新,只作为对比基准
def train_model_dpo_simple(
    policy_model, reference_model, train_loader, val_loader,
    optimizer, num_epochs, beta,
    eval_freq, eval_iter, start_context, tokenizer
):

    # Initialize lists to track losses and tokens seen
    # 用字典记录训练/验证过程中的损失、chosen/rejected 奖励以及已处理 token 数,便于后续画图分析
    tracking = {
        "train_losses": [],
        "train_chosen_rewards": [],
        "train_rejected_rewards": [],
        "val_losses": [],
        "val_chosen_rewards": [],
        "val_rejected_rewards": [],
        "tokens_seen": []
    }
    tokens_seen, global_step = 0, -1

    # Main training loop
    # 按 epoch 遍历数据
    for epoch in range(num_epochs):
        policy_model.train()  # Set model to training mode

        # 按 batch 遍历训练集
        for batch in train_loader:

            optimizer.zero_grad()  # Reset loss gradients from previous batch iteration

            # 前向计算当前 batch 的 DPO 损失(标量),以及用于监控的 chosen/rejected 隐式奖励
            loss, chosen_rewards, rejected_rewards = compute_dpo_loss_batch(
                batch=batch,
                policy_model=policy_model,
                reference_model=reference_model,
                beta=beta
            )

            loss.backward()  # Calculate loss gradients
            optimizer.step()  # Update model weights using loss gradients

            # 累计已处理的 token 数量(chosen 张量元素个数),用于后续按 token 数绘制训练曲线
            tokens_seen += batch["chosen"].numel()
            global_step += 1

            # Optional evaluation step
            # 每隔 eval_freq 步,在训练集和验证集上各采样 eval_iter 个 batch 做一次快速评估,并记录指标
            if global_step % eval_freq == 0:
                res = evaluate_dpo_loss_loader(
                    policy_model=policy_model,
                    reference_model=reference_model,
                    train_loader=train_loader,
                    val_loader=val_loader,
                    beta=beta,
                    eval_iter=eval_iter
                )
                tracking["train_losses"].append(res["train_loss"])
                tracking["train_chosen_rewards"].append(res["train_chosen_reward"])
                tracking["train_rejected_rewards"].append(res["train_rejected_reward"])
                tracking["val_losses"].append(res["val_loss"])
                tracking["val_chosen_rewards"].append(res["val_chosen_reward"])
                tracking["val_rejected_rewards"].append(res["val_rejected_reward"])
                tracking["tokens_seen"].append(tokens_seen)
                train_reward_margin = res["train_chosen_reward"] - res["train_rejected_reward"]
                val_reward_margin = res["val_chosen_reward"] - res["val_rejected_reward"]

                print(
                    f"Ep {epoch+1} (Step {global_step:06d}): "
                    f"Train loss {res['train_loss']:.3f}, Val loss {res['val_loss']:.3f}, "
                    f"Train reward margins {train_reward_margin:.3f}, "
                    f"Val reward margins {val_reward_margin:.3f}"
                )

        # 每个 epoch 结束后生成一段示例文本,直观感受当前 policy 模型的输出变化。
        # 注意: 这里传入的是全局变量 model,而不是本函数的参数 policy_model;
        # 二者当前指向同一个模型对象(见前面 policy_model = model),因此行为一致,
        # 但更严谨的写法应改用 policy_model 参数(风险项,未改动原代码,仅作提示)
        # Print a sample text after each epoch
        generate_and_print_sample(
            model=model,
            tokenizer=tokenizer,
            device=loss.device,
            start_context=start_context
        )

    # 训练函数定义结束
    return tracking
# 注意: 以下代码与上面的函数定义同处一个 cell,但属于顶层(非缩进)代码——
# 这里是在正式训练之前,先跑一次基线评估,记录“训练前”的 DPO 损失和奖励margin,便于和训练后的结果作对比
torch.manual_seed(123) # For reproducibility due to the shuffling in the data loader

res = evaluate_dpo_loss_loader(
    policy_model=policy_model,
    reference_model=reference_model,
    train_loader=train_loader,
    val_loader=val_loader,
    beta=0.1,
    eval_iter=5
)

print("Training loss:", res["train_loss"])
print("Validation loss:", res["val_loss"])

print("Train reward margin:", res["train_chosen_reward"] - res["train_rejected_reward"])
print("Val reward margin:", res["val_chosen_reward"] - res["val_rejected_reward"])

In [ ]:
# 训练前的基线展示: 在训练开始前,用验证集前 3 条样本看看当前(SFT)模型的真实生成效果
torch.manual_seed(123)


for entry in val_data[:3]:

    input_text = format_input(entry)

    # 用当前 model(此时仍是 DPO 训练前的 SFT 权重)生成回答
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
)

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response:\n>> {response_text.strip()}")
    print("\n-------------------------------------\n")

In [ ]:
# 正式开始 DPO 训练
import time

start_time = time.time()

torch.manual_seed(123)


# 只优化 policy_model 的参数(reference_model 不参与优化器),学习率设置得很小(5e-6),
# 因为是在已经收敛的 SFT 模型基础上做偏好对齐微调,过大的学习率容易破坏已有能力
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=5e-6, weight_decay=0.01)

# 只跑 1 个 epoch;beta=0.1 控制 policy 与 reference 之间的偏离程度;
# eval_freq/eval_iter 控制每隔多少步、用多少个 batch 做一次评估
num_epochs = 1
tracking = train_model_dpo_simple(
    policy_model=policy_model,
    reference_model=reference_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    num_epochs=num_epochs,
    beta=0.1, # value between 0.1 and 0.5
    eval_freq=5,
    eval_iter=5,
    start_context=format_input(val_data[2]),
    tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
# 绘制训练过程中 DPO 损失(train/val)随训练进度变化的曲线
from previous_chapters import plot_losses
# Alternatively:
# from llms_from_scratch.ch05 import plot_losses


# 构造与记录点数量匹配的“已训练 epoch 数”坐标轴,用于横轴对齐
epochs_tensor = torch.linspace(0, num_epochs, len(tracking["train_losses"]))
plot_losses(
    epochs_seen=epochs_tensor,
    tokens_seen=tracking["tokens_seen"],
    train_losses=tracking["train_losses"],
    val_losses=tracking["val_losses"],
    label="loss"
)

In [ ]:
# 奖励 margin = chosen_reward - rejected_reward,即“模型认为 chosen 比 rejected 好多少”;
# 该值随训练上升,说明 policy 模型越来越能区分并偏向 chosen 回答
train_reward_margins = [i-j for i,j in zip(tracking["train_chosen_rewards"], tracking["train_rejected_rewards"])]
val_reward_margins = [i-j for i,j in zip(tracking["val_chosen_rewards"], tracking["val_rejected_rewards"])]

plot_losses(
    epochs_seen=epochs_tensor,
    tokens_seen=tracking["tokens_seen"],
    train_losses=train_reward_margins,
    val_losses=val_reward_margins,
    label="reward margins"
)

In [ ]:
# 训练后对比: 在验证集上,同时用 reference_model(训练前基线)与 policy_model(DPO 训练后)生成回答,直观比较差异
torch.manual_seed(123)


for entry in val_data[:3]:

    input_text = format_input(entry)

    # 用未更新过的 reference_model 生成回答,作为对照
    token_ids = generate(
        model=reference_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    reference_response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    # 用 DPO 训练后的 policy_model 生成回答
    token_ids = generate(
        model=policy_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    policy_response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nReference model response:\n>> {reference_response_text.strip()}")
    print(f"\nPolicy model response:\n>> {policy_response_text.strip()}")
    print("\n-------------------------------------\n")

In [ ]:
# 在测试集上重复同样的对比,进一步确认 DPO 训练后的效果提升是否具有泛化性(而非只在验证集上过拟合)
torch.manual_seed(123)


for entry in test_data[:3]:

    input_text = format_input(entry)

    # reference_model(基线)生成
    token_ids = generate(
        model=reference_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    reference_response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    # policy_model(DPO 后)生成
    token_ids = generate(
        model=policy_model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    policy_response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nReference model response:\n>> {reference_response_text.strip()}")
    print(f"\nPolicy model response:\n>> {policy_response_text.strip()}")
    print("\n-------------------------------------\n")